# Data Loading and Merging

Merges the two single-channel pickles (`df_ch1.pkl`, `df_ch2.pkl`) into one frame.

Outputs `processed_data.pkl`.

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})

FILE_CH1 = 'df_ch1.pkl'
FILE_CH2 = 'df_ch2.pkl'
OUTPUT   = 'processed_data.pkl'

## Load Raw Channel Pickled Data

In [ ]:
with open(FILE_CH1, 'rb') as f:
    df_ch1 = pickle.load(f)
with open(FILE_CH2, 'rb') as f:
    df_ch2 = pickle.load(f)

print(f'CH1 shape : {df_ch1.shape}')
print(f'CH2 shape : {df_ch2.shape}')
print(f'CH1 columns: {df_ch1.columns.tolist()}')
display(df_ch1.head(3))

In [ ]:
# Sanity: timestamp overlap
ts1, ts2 = set(df_ch1['timestamp']), set(df_ch2['timestamp'])
print(f'Exact timestamp matches : {len(ts1 & ts2):,}')
print(f'CH1-only timestamps     : {len(ts1 - ts2):,}')
print(f'CH2-only timestamps     : {len(ts2 - ts1):,}')

n_per_sweep_ch1 = len(df_ch1['data'].iloc[0])
n_per_sweep_ch2 = len(df_ch2['data'].iloc[0])
print(f'\nPoints per sweep — CH1 : {n_per_sweep_ch1}   CH2 : {n_per_sweep_ch2}')

## Merge by Timestamp

The `data` columns are renamed to `features_a` / `features_b` for consistency with trial_02.

In [ ]:
keep_meta = ['gas_type', 'concentration_ul', 'concentration_ppm',
             'timestamp', 'temperature', 'relative_humidity']

df_a = df_ch1[keep_meta + ['data']].rename(columns={'data': 'features_a'})
df_b = df_ch2[['timestamp', 'data']].rename(columns={'data': 'features_b'})

df = df_a.merge(df_b, on='timestamp', how='inner').reset_index(drop=True)

print(f'Merged shape : {df.shape}')
print(f'Columns      : {df.columns.tolist()}')
display(df.head(3))

## Dataset Overview

In [ ]:
df_pos = df[df['concentration_ul'] > 0].copy()
print('--- After removing zero-concentration rows ---')
print(f'Total sweeps : {len(df_pos):,}')
print(f'Gas types    : {sorted(df_pos["gas_type"].unique())}')
print(f'Conc ppm     : [{df_pos["concentration_ppm"].min():.1f}, {df_pos["concentration_ppm"].max():.1f}]')
print()
print(df_pos.groupby('gas_type')['concentration_ul'].describe()[['count','min','max']].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

# Sweep count per gas type (non-zero only)
counts = df_pos['gas_type'].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Gas type')
axes[0].set_ylabel('Sweep count')
axes[0].set_title('Sweeps per gas type (non-zero conc.)')
axes[0].tick_params(axis='x', rotation=45)

# Concentration ppm distribution
axes[1].hist(df_pos['concentration_ppm'], bins=50, edgecolor='black', linewidth=0.3)
axes[1].set_xlabel('Concentration (ppm)')
axes[1].set_ylabel('Count')
axes[1].set_title('Concentration distribution (non-zero conc.)')

plt.tight_layout()
plt.show()

In [ ]:
# Spot-check: overlay a few raw sweeps per sensor
n_pts = len(df['features_a'].iloc[0])
freq_ghz = np.linspace(2, 8, n_pts)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for gas in sorted(df_pos['gas_type'].unique()):
    sample = df_pos[df_pos['gas_type'] == gas].iloc[0]
    axes[0].plot(freq_ghz, sample['features_a'], alpha=0.7, lw=0.7, label=gas)
    axes[1].plot(freq_ghz, sample['features_b'], alpha=0.7, lw=0.7, label=gas)

for ax, title in zip(axes, ['Sensor A — GO/Nafion (S11)', 'Sensor B — G/GO/PEDOT:PSS (S22)']):
    ax.set_xlabel('Frequency (GHz)')
    ax.set_ylabel('S11 (a.u.)')
    ax.set_title(title)

axes[1].legend(title='Gas', fontsize=7, loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

## Export Combined Dataset

In [ ]:
with open(OUTPUT, 'wb') as f:
    pickle.dump(df, f)
print(f'Saved {OUTPUT}  ({df.shape[0]:,} rows x {df.shape[1]} cols)')
print('Ready for 01_preliminary.ipynb')